# Benchmark MDE su DIODE Outdoor — FastDepthV2 (PyTorch)

Notebook di valutazione per modelli di **Monocular Depth Estimation (MDE)** sul subset **outdoor** di DIODE.
Pensato come **template riproducibile**: per testare un altro modello si modifica **solo l'adapter** (Cella 4),
lasciando invariati loader, allineamento e metriche, così che i risultati siano direttamente confrontabili.
Questo notebook è la **copia per FastDepthV2** del template MiDaS: dataset, range, allineamento, metriche, loop,
dashboard e report sono identici; cambiano solo l'adapter del modello (Cella 4) e l'identità del modello (Cella 2).

**Dataset:** DIODE Outdoor (`val/outdoor` — scene_00022/00023/00024), depth in metri + mask di validità.
**Range di valutazione:** 0.6 m – 20.0 m (parametrizzato, identico per tutti i modelli).
**Internet OFF:** tutto caricato da `/kaggle/input/`.

**Metodologia (identica per ogni modello — requisito di confrontabilità):**
- FastDepth produce **profondità metrica** (m); viene convertita in disparità (inverse depth) e allineata con lo
  **stesso** protocollo scala+shift usato per MiDaS, così i due modelli sono confrontati a parità di condizioni.
- Due protocolli di allineamento *per immagine*:
  - **LSQ scale+shift** in spazio disparità (protocollo accademico).
  - **Median scaling** (scala-only) in spazio disparità (protocollo deployment).
- Pixel validi: `mask==1 & MIN ≤ depth ≤ MAX`.
- Metriche: AbsRel, SqRel, RMSE, RMSE_log, δ<1.25, δ<1.25², δ<1.25³ — mediate per immagine.

**Da NON modificare tra notebook diversi** (per restare confrontabili): `MIN_DEPTH`, `MAX_DEPTH`, `N_IMAGES`,
il loader `DIODEOutdoorDataset`, le funzioni di allineamento e `compute_metrics`.
**Da adattare per ogni modello:** la classe adapter (`load`/`preprocess`/`infer`) e l'attributo `output_type`.

---
**Output — `/kaggle/working/` (ricreato da zero a ogni run):**
- `results_<slug>.json` — metriche aggregate in formato machine-readable (per il notebook di confronto).
- `TEST_OUTPUT/eval/` — `metrics_per_image.csv` + copia del results.json.
- `TEST_OUTPUT/comparison/` — `dashboard_<slug>.png` (riepilogo a 6 pannelli).
- `TEST_OUTPUT/markdown/` — `report_<slug>.md` e `metrics_<slug>.md` (leggibili da IA).

---
**Pipeline:**
1. STEP 0 — Ispezione struttura dataset e modello
2. Config (parametri di benchmark fissi + identità modello + cartelle output)
3. `DIODEOutdoorDataset` loader (solo outdoor)
4. Adapter del modello ← **unica parte da cambiare**
5. Allineamento scala+shift (LSQ + Median)
6. Metriche depth standard
7. Loop valutazione (metriche per immagine)
8. `results_<slug>.json`
9. Tabella riepilogativa (console)
10. Dashboard riepilogo (PNG)
11. Report Markdown (leggibile da IA)

In [ ]:
# ============================================================
# Cella 1 — Import + STEP 0: ispezione struttura
# ============================================================
import os
import glob
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

warnings.filterwarnings('ignore')

# --- STEP 0: albero dataset (2-3 livelli) ---
DATASET_ROOT = Path("/kaggle/input/datasets/artemmmtry/diode-a-dense-indoor-and-outdoor-depth-dataset")
OUTDOOR_ROOT = DATASET_ROOT / "val" / "outdoor"

print("=" * 60)
print("STEP 0 — Ispezione struttura dataset")
print("=" * 60)
print(f"\nDataset root esiste: {DATASET_ROOT.exists()}")
print(f"Outdoor root esiste: {OUTDOOR_ROOT.exists()}")

# Albero 3 livelli
print("\n--- Albero dataset (max 3 livelli) ---")
for lvl0 in sorted(DATASET_ROOT.iterdir()):
    print(f"  {lvl0.name}/")
    if lvl0.is_dir():
        for lvl1 in sorted(lvl0.iterdir()):
            print(f"    {lvl1.name}/")
            if lvl1.is_dir():
                for lvl2 in sorted(lvl1.iterdir()):
                    print(f"      {lvl2.name}/")

# Conferma terne .png / _depth.npy / _depth_mask.npy
print("\n--- Verifica terne (png + depth.npy + mask.npy) in OUTDOOR ---")
png_files  = sorted(glob.glob(str(OUTDOOR_ROOT / "**" / "*.png"),            recursive=True))
depth_npy  = sorted(glob.glob(str(OUTDOOR_ROOT / "**" / "*_depth.npy"),      recursive=True))
mask_npy   = sorted(glob.glob(str(OUTDOOR_ROOT / "**" / "*_depth_mask.npy"), recursive=True))
print(f"  .png trovati:            {len(png_files)}")
print(f"  _depth.npy trovati:      {len(depth_npy)}")
print(f"  _depth_mask.npy trovati: {len(mask_npy)}")
if png_files:
    print(f"  Esempio:  {Path(png_files[0]).name}")
    d0 = np.load(depth_npy[0])
    m0 = np.load(mask_npy[0])
    img0 = cv2.imread(png_files[0])
    print(f"  Shape RGB: {img0.shape}")
    print(f"  Shape depth.npy: {d0.shape}  dtype={d0.dtype}  range=[{d0.min():.3f}, {d0.max():.3f}]")
    print(f"  Shape mask.npy:  {m0.shape}  dtype={m0.dtype}  valori unici={np.unique(m0)}")

In [ ]:
# ============================================================
# Cella 2 — Configurazione
# ============================================================

# --- Identità del modello (DA CAMBIARE per ogni notebook di confronto) ---
MODEL_NAME = "FastDepthV2 (L1+GN, PyTorch)"
MODEL_SLUG = "fastdepth"   # usato per i nomi dei file di output

# --- Parametri di benchmark FISSI (devono restare IDENTICI tra i notebook) ---
MIN_DEPTH = 0.6     # metri
MAX_DEPTH = 20.0    # metri
N_IMAGES  = 200     # campione (None = tutto il dataset). Stesso valore per tutti i modelli.

# --- Path modello (specifico del modello) ---
# >>> ADATTA QUESTO PATH alla posizione del checkpoint <<<
FASTDEPTH_CKPT_PATH  = Path("/kaggle/input/models/lorenzoverdura/fastdepth/pytorch/default/1/FastDepthV2_L1GN_Best.pth")
FASTDEPTH_INPUT_SIZE = 224   # risoluzione di training NYU (input quadrato 224x224)

# --- Path output (struttura TEST_OUTPUT, ricreata da zero a ogni run) ---
OUTPUT_DIR     = Path("/kaggle/working")
TEST_OUTPUT    = OUTPUT_DIR / "TEST_OUTPUT"
DIR_EVAL       = TEST_OUTPUT / "eval"          # metriche per-immagine (CSV)
DIR_COMPARISON = TEST_OUTPUT / "comparison"    # dashboard riepilogo (PNG)
DIR_MARKDOWN   = TEST_OUTPUT / "markdown"      # report .md leggibili da IA

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if TEST_OUTPUT.exists():
    shutil.rmtree(TEST_OUTPUT)
for d in (DIR_EVAL, DIR_COMPARISON, DIR_MARKDOWN):
    d.mkdir(parents=True, exist_ok=True)

print(f"MODEL_NAME:       {MODEL_NAME}")
print(f"MODEL_SLUG:       {MODEL_SLUG}")
print(f"MIN_DEPTH:        {MIN_DEPTH} m")
print(f"MAX_DEPTH:        {MAX_DEPTH} m")
print(f"N_IMAGES:         {N_IMAGES}")
print(f"FastDepth ckpt:   {FASTDEPTH_CKPT_PATH}  —  esiste: {FASTDEPTH_CKPT_PATH.exists()}")
print(f"Input size:       {FASTDEPTH_INPUT_SIZE}x{FASTDEPTH_INPUT_SIZE}")
print(f"Output dir:       {TEST_OUTPUT}")

In [ ]:
# ============================================================
# Cella 3 — DIODEDataset loader (solo outdoor)
# ============================================================

class DIODEOutdoorDataset:
    """
    Carica terne (RGB, depth, mask) dal subset OUTDOOR di DIODE.
    Struttura attesa:
      outdoor/scene_XXXXX/scan_XXXXX/<nome>.png
                                     <nome>_depth.npy
                                     <nome>_depth_mask.npy
    Restituisce:
      rgb   : uint8  (H, W, 3)
      depth : float32 (H, W)  in metri
      mask  : bool   (H, W)   True = pixel valido

    NOTA confrontabilità: l'ordinamento delle terne è deterministico (sorted),
    quindi con lo stesso N_IMAGES tutti i notebook valutano le STESSE immagini.
    """

    def __init__(self, outdoor_root: Path, n: int = None):
        self.samples = self._find_samples(outdoor_root)
        if n is not None:
            self.samples = self.samples[:n]
        print(f"DIODEOutdoorDataset: {len(self.samples)} terne trovate")

    @staticmethod
    def _find_samples(root: Path):
        """
        Raccoglie terne ricorsivamente:
          <name>.png + <name>_depth.npy + <name>_depth_mask.npy
        """
        samples = []
        for png in sorted(root.rglob("*.png")):
            stem  = png.stem
            d_npy = png.parent / f"{stem}_depth.npy"
            m_npy = png.parent / f"{stem}_depth_mask.npy"
            if d_npy.exists() and m_npy.exists():
                samples.append((png, d_npy, m_npy))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        png_path, d_path, m_path = self.samples[idx]

        # RGB
        img_bgr = cv2.imread(str(png_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Impossibile leggere: {png_path}")
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Depth (metri) — squeeze se shape (H,W,1)
        depth = np.load(str(d_path)).astype(np.float32)
        if depth.ndim == 3:
            depth = depth.squeeze(axis=-1)

        # Mask — squeeze se shape (H,W,1)
        mask_raw = np.load(str(m_path))
        if mask_raw.ndim == 3:
            mask_raw = mask_raw.squeeze(axis=-1)
        mask = mask_raw.astype(bool)

        return rgb, depth, mask


dataset = DIODEOutdoorDataset(OUTDOOR_ROOT, n=N_IMAGES)

# Verifica campione
rgb0, depth0, mask0 = dataset[0]
print(f"Campione [0]: rgb={rgb0.shape} depth={depth0.shape} mask={mask0.shape}")
print(f"  depth range (tutti pixel): [{depth0.min():.3f}, {depth0.max():.3f}] m")
valid0 = mask0 & (depth0 >= MIN_DEPTH) & (depth0 <= MAX_DEPTH)
print(f"  pixel validi nel range [{MIN_DEPTH},{MAX_DEPTH}] m: {valid0.sum()} / {valid0.size} "
      f"({100*valid0.mean():.1f}%)")

In [ ]:
# ============================================================
# Cella 4 — DepthModelAdapter astratto + FastDepthTorchAdapter
# ============================================================
# >>> UNICA PARTE MODEL-SPECIFIC <<<
# Per testare un altro modello MDE: scrivi una nuova classe che eredita da
# DepthModelAdapter, implementa load/preprocess/infer e imposta output_type.
# Tutto il resto del notebook (allineamento, metriche, loop) resta invariato.
from abc import ABC, abstractmethod
import torch


class DepthModelAdapter(ABC):
    """
    Interfaccia astratta per adapter di modelli depth.

    output_type indica COSA produce il modello, e determina come l'allineamento
    converte l'output in profondità metrica (vedi Cella 5):
      - "affine_inverse" : disparità / inverse depth affine-invariante (MiDaS, DPT, ...)
      - "metric_depth"   : profondità in metri (FastDepth, ZoeDepth, Metric3D, ...)
      - "affine_depth"   : profondità affine-invariante (scala/shift ignoti)
    """
    output_type = "affine_inverse"

    @abstractmethod
    def load(self):
        """Carica il modello da disco."""

    @abstractmethod
    def preprocess(self, rgb: np.ndarray) -> np.ndarray:
        """RGB uint8 (H,W,3) -> input_tensor pronto per il modello."""

    @abstractmethod
    def infer(self, input_tensor: np.ndarray, target_hw: tuple = None) -> np.ndarray:
        """input_tensor -> mappa grezza (H, W) float32, ridimensionata a target_hw."""


# Costanti preprocessing FastDepth
# FastDepth è addestrato su NYU Depth v2 con RGB normalizzato in [0,1]
# (nessuna normalizzazione ImageNet nella pipeline originale).
_FASTDEPTH_INPUT_SIZE = 224   # input quadrato 224x224


class FastDepthTorchAdapter(DepthModelAdapter):
    """
    Adapter per FastDepthV2 — checkpoint PyTorch .pth.

    load() rileva da solo cosa contiene il file:
      - TorchScript        -> caricato con torch.jit.load (file autosufficiente).
      - Modello completo   -> torch.save(model): usato direttamente (serve però la
                              classe importabile).
      - state_dict (pesi)  -> NON caricabile da solo: serve la definizione
                              dell'architettura FastDepthV2. In questo caso load()
                              stampa le chiavi trovate e solleva un errore esplicito.

    output_type = "metric_depth": depth in metri, convertita in disparità e
    allineata scala+shift dallo STESSO codice usato per MiDaS (Cella 5) -> confronto equo.
    Input:  NCHW float32 in [0,1]  (1, 3, H, W).
    Output: depth (H_orig, W_orig) dopo resize bilineare al GT.
    """
    output_type = "metric_depth"

    def __init__(self, ckpt_path: Path, input_size: int = _FASTDEPTH_INPUT_SIZE, device: str = None):
        self.ckpt_path = str(ckpt_path)
        self.input_h   = int(input_size)
        self.input_w   = int(input_size)
        self.device    = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model     = None
        self.kind      = None   # "torchscript" | "module"

    def load(self):
        # Caso 1 — TorchScript: file autosufficiente, nessun codice architettura.
        try:
            self.model = torch.jit.load(self.ckpt_path, map_location=self.device).eval()
            self.kind  = "torchscript"
            print(f"FastDepthV2 caricato  (TorchScript, device={self.device})")
            print(f"  Input atteso: (1, 3, {self.input_h}, {self.input_w})  dtype=float32  norm=[0,1]")
            return
        except Exception:
            pass

        # Caso 2/3 — pickle: modello completo oppure solo state_dict.
        try:
            obj = torch.load(self.ckpt_path, map_location=self.device, weights_only=False)
        except TypeError:
            obj = torch.load(self.ckpt_path, map_location=self.device)

        # Estrai l'oggetto utile se è incapsulato in un dict di checkpoint.
        if isinstance(obj, dict):
            for k in ("model", "state_dict", "model_state_dict", "net", "weights"):
                if k in obj and obj[k] is not None:
                    obj = obj[k]
                    break

        if isinstance(obj, torch.nn.Module):
            self.model = obj.to(self.device).eval()
            self.kind  = "module"
            n_par = sum(p.numel() for p in self.model.parameters())
            print(f"FastDepthV2 caricato  (modello completo, device={self.device})")
            print(f"  Classe: {type(self.model).__name__}  —  parametri: {n_par/1e6:.2f}M")
            print(f"  Input atteso: (1, 3, {self.input_h}, {self.input_w})  dtype=float32  norm=[0,1]")
            return

        # Caso 3 — solo pesi: serve l'architettura FastDepthV2 per istanziare la rete.
        keys = list(obj.keys()) if hasattr(obj, "keys") else []
        raise RuntimeError(
            "Il checkpoint contiene SOLO uno state_dict (pesi), non il modello completo.\n"
            "Per caricarlo serve la definizione dell'architettura FastDepthV2 (file .py / repo):\n"
            "  net = FastDepthV2(...);  net.load_state_dict(<state_dict>);  net.eval()\n"
            f"  Parametri/chiavi trovate: {len(keys)}\n"
            f"  Prime chiavi: {keys[:8]}"
        )

    def preprocess(self, rgb: np.ndarray) -> np.ndarray:
        """
        rgb uint8 (H,W,3) -> NCHW float32 in [0,1]  (1, 3, H, W).
        """
        resized = cv2.resize(rgb, (self.input_w, self.input_h),
                             interpolation=cv2.INTER_LINEAR)
        tensor = resized.astype(np.float32) / 255.0           # (H, W, 3)
        tensor = np.transpose(tensor, (2, 0, 1))[np.newaxis]   # (1, 3, H, W)
        return np.ascontiguousarray(tensor)

    def infer(self, input_tensor: np.ndarray, target_hw: tuple = None) -> np.ndarray:
        """
        Esegue inferenza e restituisce la depth (m) ridimensionata a target_hw (H, W).
        Se target_hw è None, restituisce la risoluzione nativa del modello.
        """
        with torch.no_grad():
            x   = torch.from_numpy(input_tensor).to(self.device)
            out = self.model(x)

        # Normalizza shape a (H, W) — gestisce (1,1,H,W), (1,H,W,1), (1,H,W)
        raw = out.detach().cpu().numpy().squeeze()
        if raw.ndim != 2:
            raise ValueError(f"Output shape inatteso dopo squeeze: {raw.shape}")

        if target_hw is not None:
            raw = cv2.resize(raw, (target_hw[1], target_hw[0]),
                             interpolation=cv2.INTER_LINEAR)
        return raw.astype(np.float32)


adapter = FastDepthTorchAdapter(FASTDEPTH_CKPT_PATH, FASTDEPTH_INPUT_SIZE)
adapter.load()

# Test su campione 0
print("\nTest inferenza su campione [0]...")
_inp = adapter.preprocess(rgb0)
_disp = adapter.infer(_inp, target_hw=depth0.shape)
print(f"  output_type: {adapter.output_type}")
print(f"  Output grezzo: shape={_disp.shape}  range=[{_disp.min():.4f}, {_disp.max():.4f}]")

In [ ]:
# ============================================================
# Cella 5 — Allineamento scala+shift: LSQ e Median Scaling
# ============================================================
# Funzioni MODEL-AGNOSTIC: portano qualunque output (disparità o depth) nello
# spazio disparità e applicano lo STESSO allineamento, garantendo metriche
# confrontabili tra modelli diversi. Non modificare tra un notebook e l'altro.

_ALIGN_EPS = 1e-6


def _to_disparity(pred_raw: np.ndarray, output_type: str) -> np.ndarray:
    """
    Porta l'output del modello nello spazio disparità (inverse depth),
    così che LSQ/Median operino in modo identico per ogni modello.
      - affine_inverse : già disparità -> usato direttamente
      - metric_depth / affine_depth : depth (m) -> disparità = 1/depth
    """
    p = pred_raw.astype(np.float64)
    if output_type == "affine_inverse":
        return p
    if output_type in ("metric_depth", "affine_depth"):
        return 1.0 / np.clip(p, _ALIGN_EPS, None)
    raise ValueError(f"output_type non gestito: {output_type}")


def align_lsq(pred_raw, gt_depth, valid, output_type="affine_inverse"):
    """
    Allineamento Least-Squares scale+shift in spazio disparità (protocollo accademico).
    Trova s, t: min ||s * disp + t - (1/gt_depth)||_2  sui pixel validi.
    Restituisce: pred_depth = 1 / clip(s*disp + t, eps, inf)
    """
    disp    = _to_disparity(pred_raw, output_type)
    p       = disp[valid]
    gt_disp = 1.0 / np.clip(gt_depth[valid].astype(np.float64), _ALIGN_EPS, None)

    A = np.stack([p, np.ones_like(p)], axis=1)            # (N, 2)
    (s, t), *_ = np.linalg.lstsq(A, gt_disp, rcond=None)

    aligned_disp = s * disp + t
    pred_depth   = 1.0 / np.clip(aligned_disp, _ALIGN_EPS, None)
    return pred_depth.astype(np.float32), float(s), float(t)


def align_median(pred_raw, gt_depth, valid, output_type="affine_inverse"):
    """
    Allineamento Median Scaling scala-only in spazio disparità (protocollo deployment).
    s = median(1/gt_valid) / median(disp_valid);  pred_depth = 1 / (s * disp)
    """
    disp    = _to_disparity(pred_raw, output_type)
    p       = disp[valid]
    gt_disp = 1.0 / np.clip(gt_depth[valid].astype(np.float64), _ALIGN_EPS, None)

    s = np.median(gt_disp) / (np.median(p) + _ALIGN_EPS)

    aligned_disp = s * disp
    pred_depth   = 1.0 / np.clip(aligned_disp, _ALIGN_EPS, None)
    return pred_depth.astype(np.float32), float(s)


# Test rapido su campione 0
valid0_range = mask0 & (depth0 >= MIN_DEPTH) & (depth0 <= MAX_DEPTH)
pred_lsq0, s0, t0 = align_lsq(_disp, depth0, valid0_range, adapter.output_type)
pred_med0, sm0    = align_median(_disp, depth0, valid0_range, adapter.output_type)
print(f"Campione [0] — LSQ:    s={s0:.4f}  t={t0:.4f}")
print(f"Campione [0] — Median: s={sm0:.4f}")

In [ ]:
# ============================================================
# Cella 6 — compute_metrics
# ============================================================

def compute_metrics(pred_depth: np.ndarray, gt_depth: np.ndarray,
                    valid: np.ndarray,
                    min_d: float = MIN_DEPTH, max_d: float = MAX_DEPTH) -> dict:
    """
    Metriche standard depth sui soli pixel validi nel range [min_d, max_d].
    Predizione e GT vengono clampate a [min_d, max_d] prima del calcolo
    (convenzione standard, es. Monodepth2): limita gli outlier in modo
    identico per tutti i modelli -> metriche confrontabili.

    Metriche:
      AbsRel  = mean(|pred-gt| / gt)
      SqRel   = mean((pred-gt)^2 / gt)
      RMSE    = sqrt(mean((pred-gt)^2))
      RMSE_log= sqrt(mean((log(pred)-log(gt))^2))
      delta1  = mean(max(pred/gt, gt/pred) < 1.25)
      delta2  = mean(max(pred/gt, gt/pred) < 1.25^2)
      delta3  = mean(max(pred/gt, gt/pred) < 1.25^3)
    """
    mask_range = valid & (gt_depth >= min_d) & (gt_depth <= max_d)
    if mask_range.sum() == 0:
        nan = float('nan')
        return dict(abs_rel=nan, sq_rel=nan, rmse=nan, rmse_log=nan,
                    delta1=nan, delta2=nan, delta3=nan, valid_pixels=0)

    gt   = np.clip(gt_depth[mask_range].astype(np.float64),   min_d, max_d)
    pred = np.clip(pred_depth[mask_range].astype(np.float64), min_d, max_d)

    diff     = pred - gt
    abs_rel  = float(np.mean(np.abs(diff) / gt))
    sq_rel   = float(np.mean(diff**2 / gt))
    rmse     = float(np.sqrt(np.mean(diff**2)))
    rmse_log = float(np.sqrt(np.mean((np.log(pred) - np.log(gt))**2)))

    ratio    = np.maximum(pred / gt, gt / pred)
    delta1   = float(np.mean(ratio < 1.25))
    delta2   = float(np.mean(ratio < 1.25**2))
    delta3   = float(np.mean(ratio < 1.25**3))

    return dict(abs_rel=abs_rel, sq_rel=sq_rel, rmse=rmse, rmse_log=rmse_log,
                delta1=delta1, delta2=delta2, delta3=delta3,
                valid_pixels=int(mask_range.sum()))


# Test
m_lsq = compute_metrics(pred_lsq0, depth0, mask0)
m_med = compute_metrics(pred_med0, depth0, mask0)
print("Campione [0] — Metriche LSQ:    ", {k: f"{v:.4f}" if isinstance(v, float) else v
                                           for k, v in m_lsq.items()})
print("Campione [0] — Metriche Median: ", {k: f"{v:.4f}" if isinstance(v, float) else v
                                           for k, v in m_med.items()})

In [ ]:
# ============================================================
# Cella 7 — Loop valutazione principale
# ============================================================

METRIC_KEYS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "delta1", "delta2", "delta3"]

per_image          = []   # un record per immagine valutata (metriche LSQ + Median)
valid_pixel_ratios = []
skipped            = 0

for idx in tqdm(range(len(dataset)), desc=f"Valutazione {MODEL_SLUG} su DIODE Outdoor"):
    try:
        rgb, depth, mask = dataset[idx]
    except Exception as e:
        print(f"  Errore lettura campione {idx}: {e}")
        skipped += 1
        continue

    # Maschera valida nel range di valutazione
    valid = mask & (depth >= MIN_DEPTH) & (depth <= MAX_DEPTH)
    valid_pixel_ratios.append(float(valid.mean()))
    if valid.sum() == 0:
        continue

    # Inferenza
    inp      = adapter.preprocess(rgb)
    pred_raw = adapter.infer(inp, target_hw=depth.shape)

    # Allineamento (stesso protocollo per ogni modello, via output_type)
    pred_lsq, _, _ = align_lsq(pred_raw, depth, valid, adapter.output_type)
    pred_med, _    = align_median(pred_raw, depth, valid, adapter.output_type)

    # Metriche
    m_lsq = compute_metrics(pred_lsq, depth, mask)
    m_med = compute_metrics(pred_med, depth, mask)
    if np.isnan(m_lsq["abs_rel"]) or np.isnan(m_med["abs_rel"]):
        continue

    rec = {"idx": idx, "valid_ratio": float(valid.mean())}
    for k in METRIC_KEYS:
        rec[f"{k}_lsq"] = m_lsq[k]
        rec[f"{k}_med"] = m_med[k]
    per_image.append(rec)

df_per_image = pd.DataFrame(per_image)
df_per_image.to_csv(DIR_EVAL / "metrics_per_image.csv", index=False)

print(f"\nCampioni processati:  {len(dataset) - skipped}")
print(f"Campioni saltati:     {skipped}")
print(f"Immagini valutate:    {len(df_per_image)}")
print(f"Valid pixel ratio medio: {np.mean(valid_pixel_ratios):.3f} "
      f"(range [{np.min(valid_pixel_ratios):.3f}, {np.max(valid_pixel_ratios):.3f}])")

# Metriche medie per immagine, per i due protocolli di allineamento
RESULTS_LSQ    = {k: float(df_per_image[f"{k}_lsq"].mean()) for k in METRIC_KEYS} if not df_per_image.empty else {}
RESULTS_MEDIAN = {k: float(df_per_image[f"{k}_med"].mean()) for k in METRIC_KEYS} if not df_per_image.empty else {}

print("\n--- Metriche medie LSQ ---")
for k in METRIC_KEYS:
    print(f"  {k:12s}: {RESULTS_LSQ.get(k, float('nan')):.4f}")

print("\n--- Metriche medie Median Scaling ---")
for k in METRIC_KEYS:
    print(f"  {k:12s}: {RESULTS_MEDIAN.get(k, float('nan')):.4f}")

delta_abs_rel = RESULTS_MEDIAN.get("abs_rel", float('nan')) - RESULTS_LSQ.get("abs_rel", float('nan'))
print(f"\nDelta AbsRel (Median - LSQ): {delta_abs_rel:+.4f}  (costo dello scaling deployment)")

In [ ]:
# ============================================================
# Cella 8 — Salvataggio results_<slug>.json
# ============================================================
# Schema fisso e model-agnostic: un futuro notebook di confronto può caricare
# tutti i results_*.json da /kaggle/working e affiancare i modelli.

results = {
    "model_name":          MODEL_NAME,
    "model_slug":          MODEL_SLUG,
    "output_type":         adapter.output_type,
    "scenario":            "outdoor_diode",
    "depth_range_m":       [MIN_DEPTH, MAX_DEPTH],
    "n_images":            len(dataset),
    "n_evaluated":         int(len(df_per_image)),
    "valid_pixel_ratio":   float(np.mean(valid_pixel_ratios)) if valid_pixel_ratios else 0.0,
    "metrics_lsq":         RESULTS_LSQ,
    "metrics_median":      RESULTS_MEDIAN,
    "delta_abs_rel_median_minus_lsq": float(delta_abs_rel),
    "note":                "Allineamento scala+shift (in spazio disparità) applicato a ogni modello prima della valutazione, per confronto equo. Metriche mediate per immagine.",
}

# results_<slug>.json in /kaggle/working (facile da raccogliere) e copia in TEST_OUTPUT
results_path = OUTPUT_DIR / f"results_{MODEL_SLUG}.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
shutil.copy(results_path, DIR_EVAL / results_path.name)

print(f"results scritto in: {results_path}")

In [ ]:
# ============================================================
# Cella 8.5 — Confronto qualitativo: Ground Truth vs Predizione
# ============================================================
# Per una singola immagine DIODE mostra RGB, depth GT (con mask di validità),
# depth predetta allineata scala+shift (LSQ) e mappa errore assoluto.
# I pixel non validi nel GT sono esclusi anche dalla predizione (confronto equo)
# e mostrati in grigio. GT e predizione usano la stessa scala colore.

VIS_IDX = 0   # indice immagine da visualizzare

rgb_v, depth_v, mask_v = dataset[VIS_IDX]
valid_v = mask_v & (depth_v >= MIN_DEPTH) & (depth_v <= MAX_DEPTH)

# Inferenza + allineamento LSQ (stesso protocollo del loop di valutazione)
inp_v            = adapter.preprocess(rgb_v)
pred_raw_v       = adapter.infer(inp_v, target_hw=depth_v.shape)
pred_lsq_v, _, _ = align_lsq(pred_raw_v, depth_v, valid_v, adapter.output_type)

# Metriche di questa immagine (per il titolo)
m_v = compute_metrics(pred_lsq_v, depth_v, mask_v)

# Mascheratura: pixel non validi -> NaN (resi in grigio dalla colormap)
gt_show   = np.where(valid_v, np.clip(depth_v,    MIN_DEPTH, MAX_DEPTH), np.nan)
pred_show = np.where(valid_v, np.clip(pred_lsq_v, MIN_DEPTH, MAX_DEPTH), np.nan)
err_show  = np.where(valid_v, np.abs(pred_lsq_v - depth_v),              np.nan)

# Scala colore comune GT/predizione per confronto diretto
vmin = MIN_DEPTH
vmax = float(np.nanpercentile(gt_show, 99))
cmap = plt.cm.viridis.copy()
cmap.set_bad("lightgray")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(
    f"DIODE Outdoor — img idx {VIS_IDX} | {MODEL_NAME} (LSQ)   "
    f"AbsRel={m_v['abs_rel']:.3f}   RMSE={m_v['rmse']:.2f} m   δ<1.25={m_v['delta1']:.3f}",
    fontsize=13, fontweight="bold")

axes[0].imshow(rgb_v)
axes[0].set_title("RGB"); axes[0].axis("off")

im1 = axes[1].imshow(gt_show, cmap=cmap, vmin=vmin, vmax=vmax)
axes[1].set_title("Ground Truth depth [m]"); axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(pred_show, cmap=cmap, vmin=vmin, vmax=vmax)
axes[2].set_title("Predizione allineata LSQ [m]"); axes[2].axis("off")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(err_show, cmap="magma")
axes[3].set_title("Errore assoluto |pred-GT| [m]"); axes[3].axis("off")
fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cella 9 — Tabella riepilogativa finale (console)
# ============================================================

rows = []
for proto, res in [("LSQ (accademico)", RESULTS_LSQ),
                   ("Median (deployment)", RESULTS_MEDIAN)]:
    rows.append({
        "Protocollo":  proto,
        "AbsRel":      f"{res.get('abs_rel', float('nan')):.4f}",
        "SqRel":       f"{res.get('sq_rel', float('nan')):.4f}",
        "RMSE":        f"{res.get('rmse', float('nan')):.4f}",
        "RMSE_log":    f"{res.get('rmse_log', float('nan')):.4f}",
        "δ<1.25":      f"{res.get('delta1', float('nan')):.4f}",
        "δ<1.25²":     f"{res.get('delta2', float('nan')):.4f}",
        "δ<1.25³":     f"{res.get('delta3', float('nan')):.4f}",
    })

df_summary = pd.DataFrame(rows).set_index("Protocollo")

print("\n" + "=" * 70)
print(f"{MODEL_NAME} | DIODE Outdoor | Range {MIN_DEPTH}–{MAX_DEPTH} m")
print("=" * 70)
print(df_summary.to_string())
print("=" * 70)
print(f"N immagini valutate:     {len(df_per_image)}")
print(f"Valid pixel ratio medio: {np.mean(valid_pixel_ratios):.3f}")
print(f"\nDelta AbsRel Median-LSQ: {delta_abs_rel:+.4f}")
print(f"  -> Costo dell'allineamento semplificato (scala-only) rispetto al protocollo accademico (scala+shift)")

In [ ]:
# ============================================================
# Cella 10 — Dashboard riepilogo finale (PNG)
# ============================================================
# Dashboard 6-pannelli: accuratezza delta, errori relativi, RMSE [m],
# distribuzione per-immagine di AbsRel e del valid_pixel_ratio, configurazione.
# Salvata in TEST_OUTPUT/comparison/.

def _annota_barre(ax, bars, fmt="{:.3f}"):
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, h, fmt.format(h),
                ha="center", va="bottom", fontsize=8)

fig = plt.figure(figsize=(16, 10))
fig.suptitle(f"{MODEL_NAME} — Riepilogo MDE su DIODE Outdoor (range {MIN_DEPTH}–{MAX_DEPTH} m)",
             fontsize=15, fontweight="bold")

w = 0.38

# Pannello 1: accuratezza delta (↑ meglio, bounded [0,1])
ax1 = fig.add_subplot(2, 3, 1)
dk = ["delta1", "delta2", "delta3"]
x  = np.arange(len(dk))
b1 = ax1.bar(x - w/2, [RESULTS_LSQ[k]    for k in dk], w, label="LSQ",    color="steelblue")
b2 = ax1.bar(x + w/2, [RESULTS_MEDIAN[k] for k in dk], w, label="Median", color="coral")
_annota_barre(ax1, list(b1) + list(b2))
ax1.set_xticks(x); ax1.set_xticklabels(["δ<1.25", "δ<1.25²", "δ<1.25³"])
ax1.set_ylim(0, 1.1); ax1.set_title("Accuratezza δ (↑ meglio)", fontsize=11); ax1.legend(fontsize=8)

# Pannello 2: errori relativi (↓ meglio)
ax2 = fig.add_subplot(2, 3, 2)
ek = ["abs_rel", "sq_rel", "rmse_log"]
x  = np.arange(len(ek))
b1 = ax2.bar(x - w/2, [RESULTS_LSQ[k]    for k in ek], w, label="LSQ",    color="steelblue")
b2 = ax2.bar(x + w/2, [RESULTS_MEDIAN[k] for k in ek], w, label="Median", color="coral")
_annota_barre(ax2, list(b1) + list(b2))
ax2.set_xticks(x); ax2.set_xticklabels(["AbsRel", "SqRel", "RMSE_log"])
ax2.set_title("Errori relativi (↓ meglio)", fontsize=11); ax2.legend(fontsize=8)

# Pannello 3: RMSE in metri (↓ meglio)
ax3 = fig.add_subplot(2, 3, 3)
b = ax3.bar(["LSQ", "Median"], [RESULTS_LSQ["rmse"], RESULTS_MEDIAN["rmse"]],
            color=["steelblue", "coral"])
_annota_barre(ax3, b, fmt="{:.3f}")
ax3.set_ylabel("metri"); ax3.set_title("RMSE [m] (↓ meglio)", fontsize=11)

# Pannello 4: distribuzione AbsRel per immagine (LSQ)
ax4 = fig.add_subplot(2, 3, 4)
ax4.hist(df_per_image["abs_rel_lsq"], bins=30, color="steelblue", edgecolor="white")
ax4.axvline(RESULTS_LSQ["abs_rel"], color="red", linestyle="--",
            label=f"media={RESULTS_LSQ['abs_rel']:.3f}")
ax4.set_xlabel("AbsRel per immagine (LSQ)"); ax4.set_ylabel("Frequenza")
ax4.set_title("Distribuzione AbsRel per immagine", fontsize=11); ax4.legend(fontsize=8)

# Pannello 5: distribuzione valid_pixel_ratio (controllo qualità)
ax5 = fig.add_subplot(2, 3, 5)
ax5.hist(valid_pixel_ratios, bins=30, color="seagreen", edgecolor="white")
ax5.axvline(np.mean(valid_pixel_ratios), color="red", linestyle="--",
            label=f"media={np.mean(valid_pixel_ratios):.3f}")
ax5.set_xlabel("valid_pixel_ratio per immagine"); ax5.set_ylabel("Frequenza")
ax5.set_title("Copertura GT valida (range)", fontsize=11); ax5.legend(fontsize=8)

# Pannello 6: info configurazione
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
info_text = (
    f"Modello:        {MODEL_NAME}\n"
    f"output_type:    {adapter.output_type}\n"
    f"Input modello:  {adapter.input_h}x{adapter.input_w}\n"
    f"Range depth:    {MIN_DEPTH}-{MAX_DEPTH} m\n"
    f"Allineamento:   LSQ scale+shift / Median\n\n"
    f"Immagini (N):     {len(dataset)}\n"
    f"Immagini valutate:{len(df_per_image)}\n"
    f"Valid px ratio:   {np.mean(valid_pixel_ratios):.3f}\n"
    f"Delta AbsRel M-L: {delta_abs_rel:+.4f}\n"
    f"Output:           TEST_OUTPUT/\n"
)
ax6.text(0.03, 0.97, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
dashboard_path = DIR_COMPARISON / f"dashboard_{MODEL_SLUG}.png"
plt.savefig(dashboard_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Dashboard salvata: {dashboard_path}")

In [ ]:
# ============================================================
# Cella 11 — Report Markdown (output leggibile da IA)
# ============================================================
# Scrive report .md in TEST_OUTPUT/markdown/: configurazione, metriche
# aggregate (LSQ vs Median), statistiche per immagine e peggiori casi.

def df_to_md(df):
    """Converte un DataFrame in tabella Markdown, senza dipendenze esterne."""
    cols = list(df.columns)
    head = "| " + " | ".join(str(c) for c in cols) + " |"
    sep  = "| " + " | ".join("---" for _ in cols) + " |"
    rows = []
    for _, r in df.iterrows():
        cells = [f"{r[c]:.4f}" if isinstance(r[c], float) else str(r[c]) for c in cols]
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join([head, sep] + rows)


# Tabella metriche aggregate: una riga per metrica, colonne LSQ e Median
METRIC_LABELS = {
    "abs_rel":  "AbsRel (↓)",
    "sq_rel":   "SqRel (↓)",
    "rmse":     "RMSE [m] (↓)",
    "rmse_log": "RMSE_log (↓)",
    "delta1":   "δ<1.25 (↑)",
    "delta2":   "δ<1.25² (↑)",
    "delta3":   "δ<1.25³ (↑)",
}
agg_df = pd.DataFrame({
    "Metrica":            [METRIC_LABELS[k] for k in METRIC_KEYS],
    "LSQ (accademico)":   [RESULTS_LSQ[k]    for k in METRIC_KEYS],
    "Median (deployment)":[RESULTS_MEDIAN[k] for k in METRIC_KEYS],
})

# Statistiche per immagine (AbsRel LSQ) e 5 immagini peggiori
abs_series = df_per_image["abs_rel_lsq"]
stats_df = pd.DataFrame({
    "Statistica (AbsRel LSQ)": ["media", "mediana", "std", "min", "max"],
    "Valore": [float(abs_series.mean()), float(abs_series.median()),
               float(abs_series.std()), float(abs_series.min()), float(abs_series.max())],
})
worst_df = (df_per_image
            .nlargest(5, "abs_rel_lsq")[["idx", "valid_ratio", "abs_rel_lsq", "rmse_lsq", "delta1_lsq"]]
            .rename(columns={"idx": "img_idx", "abs_rel_lsq": "AbsRel",
                             "rmse_lsq": "RMSE[m]", "delta1_lsq": "δ<1.25"}))

report = f"""# Report MDE — {MODEL_NAME} su DIODE Outdoor

## 1. Configurazione
| Parametro | Valore |
| --- | --- |
| Modello | {MODEL_NAME} |
| output_type | {adapter.output_type} |
| Input modello | {adapter.input_h}x{adapter.input_w} |
| Scenario | DIODE Outdoor (scene_00022/00023/00024) |
| Range depth valutato | {MIN_DEPTH}–{MAX_DEPTH} m |
| Allineamento | per immagine: LSQ scale+shift / Median scaling |
| Immagini (N) | {len(dataset)} |
| Immagini valutate | {len(df_per_image)} |
| Valid pixel ratio medio | {np.mean(valid_pixel_ratios):.4f} |

## 2. Metriche aggregate (medie per immagine)
_↓ = più basso è meglio; ↑ = più alto è meglio._
{df_to_md(agg_df)}

- **Delta AbsRel (Median − LSQ):** {delta_abs_rel:+.4f} — costo dell'allineamento scala-only (deployment) rispetto a scala+shift (accademico).

## 3. Statistiche per immagine
{df_to_md(stats_df)}

### 5 immagini con AbsRel (LSQ) peggiore
{df_to_md(worst_df)}

## 4. Note metodologiche
- L'output del modello è **{adapter.output_type}**: convertito in disparità e allineato scala+shift (stesso protocollo per tutti i modelli) prima del calcolo delle metriche, così il confronto resta equo.
- Pixel validi: `mask==1 & {MIN_DEPTH} ≤ depth ≤ {MAX_DEPTH}`. Predizione e GT clampate al range.
- Metriche mediate per immagine. Protocollo identico per tutti i modelli → risultati confrontabili.
"""

report_path = DIR_MARKDOWN / f"report_{MODEL_SLUG}.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report Markdown scritto: {report_path}")

metrics_md_path = DIR_MARKDOWN / f"metrics_{MODEL_SLUG}.md"
metrics_md_path.write_text(
    f"# Metriche aggregate — {MODEL_NAME} (DIODE Outdoor)\n\n" + df_to_md(agg_df) + "\n",
    encoding="utf-8",
)
print(f"Tabella metriche scritta: {metrics_md_path}")